In [8]:
%pip install aiohttp nest_asyncio pandas --quiet

import aiohttp
import nest_asyncio
import asyncio
import json

nest_asyncio.apply()

Note: you may need to restart the kernel to use updated packages.


In [9]:
async def fetch_gpus(filter_type="all"):
    url = "https://api.hyperbolic.xyz/v1/marketplace"
    headers = {"Content-Type": "application/json"}
    filters = {} if filter_type == "all" else {"available": True}
    data = {"filters": filters}
    async with aiohttp.ClientSession() as session:
        async with session.post(url, json=data, headers=headers) as response:
            if response.status == 200:
                return await response.json()
            else:
                print(f"API request failed with status {response.status}")
                return None

In [10]:
import pandas as pd


async def show_gpus(filter_type="all"):
    data = await fetch_gpus(filter_type)
    if data and "instances" in data:
        instances = [
            {
                "id": inst["id"],
                "gpu_model": inst["hardware"]["gpus"][0]["model"],
                "gpu_memory": inst["hardware"]["gpus"][0]["ram"],
                "price_per_hour": inst["pricing"]["price"]["amount"],
                "location": inst["location"]["region"],
                "available": not inst["reserved"]
                and inst["gpus_reserved"] < inst["gpus_total"],
            }
            for inst in data["instances"]
            if "gpus" in inst["hardware"] and inst["hardware"]["gpus"]
        ]
        df = pd.DataFrame(instances)
        display(df)
        return df
    else:
        print("No data found or API error.")


# Example usage:
df = asyncio.run(show_gpus("available_only"))

,id,gpu_model,gpu_memory,price_per_hour,location,available
0,ceti14,NVIDIA-H100-80GB-HBM3,81559,150,region-1,True
1,l-hgx-05,NVIDIA-H200,143771,220,region-1,True
2,sfc-016,NVIDIA-H100-80GB-HBM3,81559,150,region-1,True
3,l-hgx-01,NVIDIA-H200,143771,225,region-1,True
4,antalpha-super-server100132,NVIDIA-GeForce-RTX-4090,24564,35,region-1,False
...,...,...,...,...,...,...
64,antalpha-super-server100164,NVIDIA-GeForce-RTX-4090,24564,30,region-1,False
65,sfc-020,NVIDIA-H100-80GB-HBM3,81559,150,region-1,False
66,sfc-014,NVIDIA-H100-80GB-HBM3,81559,150,region-1,True
67,ceti15,NVIDIA-H100-80GB-HBM3,81559,150,region-1,False


In [11]:
if df is not None:
    df.to_csv("marketplace_output.txt", index=False, sep="\t")
    print("Saved to marketplace_output.txt")

Saved to marketplace_output.txt
